In [1]:
# -*- coding: utf-8 -*-
"""
conda install -c conda-forge keras
conda install -c conda-forge tensorflow
conda install -c anaconda scikit-learn
conda install -c pytorch pytorch

pip install scikit-multilearn

"""
import itertools

from keras.preprocessing.text import Tokenizer
from keras.models import Sequential
from keras.layers import Dense

import matplotlib.pyplot as plt

from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

from numpy import mean
from numpy import std
import numpy as np

import pandas as pd

import re

from scipy.sparse import csr_matrix, lil_matrix

# from sentence_transformers import SentenceTransformer, util

from sklearn.datasets import make_multilabel_classification
from sklearn.decomposition import TruncatedSVD
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.model_selection import RepeatedKFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import scale
from sklearn.svm import SVC

from skmultilearn.adapt import MLkNN, MLTSVM
from skmultilearn.problem_transform import BinaryRelevance
from skmultilearn.problem_transform import ClassifierChain
from skmultilearn.problem_transform import LabelPowerset

import string

import time
# import torch

from tqdm import tqdm


In [2]:
from utils.utils_logger import logger

from cria_create_indicators import ref, tracts, counties, states, create_indicators

from utils.utils_table_save import table_save

2022/03/13 14:48:59 - cria_logger - INFO - issues logger ready


In [3]:
data_ser = pd.read_pickle("data/all_data.pkl")
print(data_ser.index)

data_state = data_ser["data_state"].copy(deep=True)
data_county = data_ser["data_county"].copy(deep=True)
data_tract = data_ser["data_tract"].copy(deep=True)

list_inputs = data_state.columns
labels = data_ser["labels"].copy(deep=True)

df_state = data_ser["df_state"].copy(deep=True)
df_county = data_ser["df_county"].copy(deep=True)
df_tract = data_ser["df_tract"].copy(deep=True)

Index(['states', 'counties', 'tracts', 'df_state', 'data_state', 'df_county',
       'data_county', 'df_tract', 'data_tract', 'labels'],
      dtype='object')


In [4]:
data_county_aug = data_county.merge(counties, left_index=True, right_on="GEO_ID", how="outer")
agg_county = data_county_aug.groupby(["state"])[list_inputs].sum()

# Don't sum "index" values
cols = ref.loc[ref["Units"] == "index", "numerator"].to_list()
agg_county[cols] = data_state[cols].merge(states[["state"]], 
    left_index=True,
    right_index=True).set_index("state")
agg_county.loc["labels", :] = labels.values

df_state_aug, data_state_aug_ref  = create_indicators(pd.Series({"data": agg_county}))

2022/03/13 14:49:09 - cria_logger - INFO - 0: Education, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 1: Unemployment, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 2: Disability, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 3: Limited English, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 4: Owner Occupied, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 5: No Vehicle, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 6: Age, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 7: Median Income, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 8: GINI, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 9: Uninsured Population, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 10: Single Parent, function (divide)
2022/03/13 14:49:09 - cria_logger - INFO - 11: Inactive Voter, function (reverse_divide)
2022/03/13 14:49:09 - cria_logger - INFO - 12: Unemployed Wom

Regression to complete missing state data.  Currently, issues affect three columns
* Drop rows with missing information
* Train columns with complete information
* Test columns that had the rows with missing information

In [5]:
df_ref = df_county.copy(deep=True).loc[counties.index, :]

# Drop Rows with missing values
df = df_ref.dropna(axis=0, how="any")

# Center and Scale
df_cs = pd.DataFrame(scale(df, axis=0),
                     index=df.index,
                     columns=df.columns)

# Train, X, on columns without missing values
df_x = df_cs[df_ref.dropna(axis=1, how="any").columns]

# Predict, y, columns with missing values
df_y = df_cs.drop(df_x.columns, axis=1)

print(df_ref.shape, df.shape, df_x.shape, df_y.shape)


(3220, 22) (2909, 22) (2909, 18) (2909, 4)


In [6]:
print(df_ref.info())
# print(df_ref.corr())

<class 'pandas.core.frame.DataFrame'>
Index: 3220 entries, 0500000US01001 to 0500000US72153
Data columns (total 22 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   Education                     3220 non-null   float64
 1   Unemployment                  3220 non-null   float64
 2   Disability                    3220 non-null   float64
 3   Limited English               3220 non-null   float64
 4   Owner Occupied                3220 non-null   float64
 5   No Vehicle                    3220 non-null   float64
 6   Age                           3220 non-null   float64
 7   Median Income                 3220 non-null   float64
 8   GINI                          3220 non-null   float64
 9   Uninsured Population          3220 non-null   float64
 10  Single Parent                 3141 non-null   float64
 11  Inactive Voter                2910 non-null   float64
 12  Unemployed Women              3220 non-null 

In [7]:
data_all = pd.read_pickle("data/detailed_data.pkl")
data_all.info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 74001 entries, 0 to 74000
Columns: 263 entries, NAME_tract to Population Change_state
dtypes: float64(67), int64(5), object(191)
memory usage: 149.0+ MB


combined all data at tract level
* drop repeated name columns (look for columns that can't be coerced to numeric)
* however, some columns were mostly nan; lets track the difference
* all dropped columns stored as missing data
* of missing data, filter out list_objects
* result, missing data, is what we will try to predict

In [8]:
data = data_all.copy(deep=True)
data = data.apply(pd.to_numeric, errors="coerce")
data = data.dropna(how="all", axis=1)
print(data.shape)

cols_objs = list(set(data_all.columns)-set(data.columns))
patterns_objs = ["region", "abbr", "name"]

# ?re.search

pattern_reg = re.compile("region")
pattern_abbr = re.compile("abbr")
pattern_name = re.compile("name")
missing_data = [col for col in cols_objs if not
                 (re.search(pattern_reg, col.lower()) 
                 or re.search(pattern_abbr, col.lower()) 
                 or re.search(pattern_name, col.lower()))]
missing_data = sorted(missing_data)
print(len(missing_data), missing_data)

pattern_state = re.compile("_state")
pattern_county = re.compile("_county")
known_data = [col for col in data.columns if
              (re.search(pattern_state, str(col).lower()) 
              or re.search(pattern_county, str(col).lower()))]
known_data = sorted(known_data)

known_tract_indicators = ref.loc[ref["Source"] == "ACS", "Indicator"].values
known_tract_indicators = list(known_tract_indicators)
known_tract_indicators = [f"{col.strip()}_tract" for col in known_tract_indicators]

known_tract_data = ref.loc[ref["Source"] == "ACS", ["numerator", "denominator"]].values
known_tract_data = list(known_tract_data)

known_tract_data = [item for sublist in known_tract_data for item in sublist]
known_tract_data = [col for col in known_tract_data if type(col) == str]
known_tract_data = [col.split(", ") for col in known_tract_data]
known_tract_data = [item for sublist in known_tract_data for item in sublist]
known_tract_data = [f"{col.strip()}_tract" for col in known_tract_data if type(col)==str]
print("\n""\n")
print(known_tract_data)
print("\n""\n")

print(f"len k data {len(known_data)}, " +
      f"len k tract indicators {len(known_tract_indicators)}, " +
      f"len k tract data{len(known_tract_data)}")
print("\n""\n")

known_data = [item for sublist in 
              [known_data, known_tract_indicators, known_tract_data]
              for item in sublist]

print(len(known_data), known_data)

print(set(data.columns) - set(known_data) - set(missing_data))

(74001, 237)
12 ['A1a_tract', 'A1c_tract', 'Inactive Voter_tract', 'NETMIG2015_tract', 'NETMIG2016_tract', 'NETMIG2017_tract', 'NETMIG2018_tract', 'NETMIG2019_tract', 'POP2010_tract', 'Population Change_tract', 'Religion_tract', 'TOTADH_tract']



['S1501_C01_007E_tract', 'S1501_C01_008E_tract', 'S1501_C01_006E_tract', 'DP03_0005E_tract', 'DP03_0003E_tract', 'S1810_C02_001E_tract', 'S1810_C01_001E_tract', 'S1602_C03_001E_tract', 'S1602_C01_001E_tract', 'DP04_0046E_tract', 'DP04_0001E_tract', 'B08201_002E_tract', 'B08201_001E_tract', 'S0101_C01_015E_tract', 'S0101_C01_016E_tract', 'S0101_C01_017E_tract', 'S0101_C01_018E_tract', 'S0101_C01_019E_tract', 'S0101_C01_001E_tract', 'S1903_C03_001E_tract', 'B19083_001E_tract', 'S2701_C04_001E_tract', 'S2701_C01_001E_tract', 'DP02_0007E_tract', 'DP02_0011E_tract', 'DP02_0014E_tract', 'DP03_0013E_tract', 'DP03_0012E_tract', 'DP03_0033E_tract', 'DP03_0034E_tract', 'DP03_0035E_tract', 'DP03_0036E_tract', 'DP03_0037E_tract', 'DP03_0038E_tract', 'DP0

In [9]:
# Drop Rows with missing values
data_drp = data.dropna(axis=0, how="any")
print(data_drp.shape)

# Center and Scale
# data_cs = pd.DataFrame(scale(data_drp, axis=0),
#                      index=data_drp.index,
#                      columns=data_drp.columns)


(62549, 237)


Predict with known values

Build to iterate across missing values, record results, predict, build indicators, train tract missing information on all higher level information


In [10]:
print(missing_data[0])
col_pred = missing_data[0][:-5] + "county"
print(col_pred)
df_x = data_drp[known_data].drop(col_pred, axis=1)
df_y = data_drp[col_pred]
print(df_x.shape, df_y.shape)

A1a_tract
A1a_county
(62549, 231) (62549,)


In [11]:
# ?GridSearchCV

In [12]:
pd.isna(df_x).sum(axis=0).sort_values(ascending=False)[:15]

622110_county                  0
S2701_C04_001E_county          0
S2801_C01_001E_county          0
S2801_C01_001E_state           0
S2801_C01_005E_county          0
S2801_C01_005E_state           0
Single Parent_county           0
Single Parent_state            0
TOTADH_county                  0
TOTADH_state                   0
Unemployed Women_county        0
Unemployed Women_state         0
Unemployment_county            0
Unemployment_state             0
Uninsured Population_county    0
dtype: int64

Linear and Logistic Regression

In [13]:
X = df_x.values
y = df_y.values
res = pd.DataFrame(data=y,
                  index=df_y.index,
                  columns=["y"])
res_scores = pd.Series(dtype=object)

X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=101)

reg = LinearRegression().fit(X_train, y_train)
res_scores["lin"] = reg.score(X_test, y_test)
print(reg.score(X_test, y_test))

reg.coef_

reg.intercept_

res["y_pred_lin"] = reg.predict(X)
print(res[:10])
print(res["y_pred_lin"].nunique())


0.9993509831728608
         y    y_pred_lin
0  43695.0  48899.734131
1  43695.0  46591.717041
2  43695.0  49825.369629
3  43695.0  50810.843018
4  43695.0  65248.710205
5  43695.0  48496.860352
6  43695.0  51338.915039
7  43695.0  47736.455566
8  43695.0  49882.740723
9  43695.0  55908.645508
62548


In [19]:
print(col_pred)
# Alaska
rows = data.loc[data["state_county"]==2, col_pred].index
print(rows[:5])
print(df_x.columns[:5])
pd.isna(df_x).sum(axis=0).sort_values(ascending=False)[:15]
print(pd.isna(data.loc[rows,df_x.columns]).sum(axis=0).sort_values(ascending=False)[:15])
reg.predict(data.loc[rows,df_x.columns].fillna(0).values)[:5]
print(reg.predict(data.loc[rows,df_x.columns].fillna(0).values).sum())
print(data.loc[rows, "S0101_C01_001E_county"].sum())
print(reg.predict(data.loc[rows,df_x.columns].fillna(0).values).sum() / data.loc[rows, "S0101_C01_001E_county"].sum())
agg_data = data.groupby("state_county")[["A1c_county", "S0101_C01_001E_county"]].sum()
agg_data["comp"] = agg_data["A1c_county"] / agg_data["S0101_C01_001E_county"]
print(agg_data.head(15))
print(agg_data.describe())

A1a_county
Int64Index([1181, 1182, 1183, 1184, 1185], dtype='int64')
Index(['622110_county', '622110_state', '813410_county', '813410_state',
       'A1a_state'],
      dtype='object')
A1c_county                            167
Inactive Voter_state                  167
Inactive Voter_county                 167
Single Parent_tract                     2
Mobile Homes_tract                      1
TOTADH_county                           1
No Vehicle_tract                        1
Owner Occupied_tract                    1
Limited English_tract                   1
Low Access to Communications_tract      1
POP2010_county                          1
Religion_county                         1
Unemployed Women_county                 0
TOTADH_state                            0
622110_county                           0
dtype: int64
1756131441559373.0
21908141.0
80158852.43569379
                A1c_county  S0101_C01_001E_county      comp
state_county                                               
1   

reg = LogisticRegression().fit(X_train, y_train)
res_scores["log"] = reg.score(X_test, y_test)
print(reg.score(X_test, y_test))

reg.coef_

reg.intercept_

res["y_pred_log"] = reg.predict(X)
print(res[:10])

In [ ]:
%%time

X = df_x.values
y = df_y.values
print(X.shape, y.shape)
# prep data: train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.2,
                                                    random_state=101)

grid_layers = [x for x in itertools.product((10, 50, 100, 150, 200),repeat=4)]
len(grid_layers)

# params = {"hidden_layer_sizes": grid_layers,
#           "activation": ["identity", "logistic", "tanh", "relu"],
#           "solver":["lbfgs", "sgd", "adam"],
#           "alpha":[1e-3],
#           "learning_rate":["constant", "invscaling", "adaptive"],
#           "random_state":[101]}

params = {"hidden_layer_sizes": grid_layers,
          "activation": ["tanh"],
          "solver": ["sgd"],
          "alpha":[1e-3],
          "learning_rate":["invscaling"],
          "random_state":[101]}

grid = GridSearchCV(MLPRegressor(), params, verbose=2)
grid.fit(X_train, y_train)

(62549, 231) (62549,)
Fitting 5 folds for each of 625 candidates, totalling 3125 fits


C:\Users\johnk\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 10, 10, 10), learning_rate=invscaling, random_state=101, solver=sgd; total time=  20.0s


C:\Users\johnk\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 10, 10, 10), learning_rate=invscaling, random_state=101, solver=sgd; total time=  23.7s


...

[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 10), learning_rate=invscaling, random_state=101, solver=sgd; total time= 1.2min


C:\Users\johnk\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 10), learning_rate=invscaling, random_state=101, solver=sgd; total time= 1.1min


C:\Users\johnk\anaconda3\lib\site-packages\sklearn\neural_network\_multilayer_perceptron.py:692: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 10), learning_rate=invscaling, random_state=101, solver=sgd; total time= 1.2min
[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 10), learning_rate=invscaling, random_state=101, solver=sgd; total time=  23.8s
[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 50), learning_rate=invscaling, random_state=101, solver=sgd; total time=  20.2s
[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 50), learning_rate=invscaling, random_state=101, solver=sgd; total time=  19.7s
[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 50), learning_rate=invscaling, random_state=101, solver=sgd; total time=  23.2s
[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 50), learning_rate=invscaling, random_state=101, solver=sgd; total time=  33.5s
[CV] END activation=tanh, alpha=0.001, hidden_layer_sizes=(10, 200, 50, 50), learning_ra

In [25]:
print(X.shape, y.shape)
df_y.describe()

(62549, 231) (62549,)


count    6.254900e+04
mean     8.025752e+05
std      1.393754e+06
min      1.110000e+02
25%      8.027000e+04
50%      3.229210e+05
75%      9.459920e+05
max      7.122542e+06
Name: A1a_county, dtype: float64

In [23]:
print(grid.best_estimator_)
print(grid.best_params_)
# res_grid = pd.DataFrame(grid.cv_results_)
# res_grid.to_excel("data/res_grid_genesis.xlsx")

# Neural Net
# model, fit, predict
mlp_reg = MLPRegressor(**grid.best_params_)
mlp_reg.fit(X_train,y_train)
y_pred = mlp_reg.predict(X_test)

print("{} R^2, coef of determination of the prediction".format(mlp_reg.score(X_test, y_test)))

y_pred = pd.DataFrame(mlp_reg.predict(X))

y_reg = pd.concat([df_y, y_pred], axis=1).rename({0:"pred"}, axis=1)
print(y_reg.head())
print(y_reg.nunique(axis=0))

MLPRegressor(activation='tanh', alpha=0.001, hidden_layer_sizes=(100, 50, 50),
             learning_rate='invscaling', random_state=101, solver='sgd')
{'activation': 'tanh', 'alpha': 0.001, 'hidden_layer_sizes': (100, 50, 50), 'learning_rate': 'invscaling', 'random_state': 101, 'solver': 'sgd'}
-0.0008576118254239873 R^2, coef of determination of the prediction
   A1a_county           pred
0     43695.0  797978.824499
1     43695.0  797978.824499
2     43695.0  797978.824499
3     43695.0  797978.824499
4     43695.0  797978.824499
A1a_county    2711
pred             6
dtype: int64


In [ ]:
print(y_reg.A1a_county.nunique())

data_ref = data_county.copy(deep=True).loc[counties.index, :]

# Drop Rows with missing values
data = data_ref.dropna(axis=0, how="any")

# Center and Scale
data_cs = pd.DataFrame(scale(data, axis=0),
                     index=data.index,
                     columns=data.columns)

# Train, X, on columns without missing values
data_x = data_cs[data_ref.dropna(axis=1, how="any").columns]

# Predict, y, columns with missing values
data_y = data_cs.drop(data_x.columns, axis=1)

print(data_ref.shape, data.shape, data_x.shape, data_y.shape)

%%time
X = data_x.values
y = data_y.values
# prep data: train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.1,
                                                    random_state=101)

grid_layers = [x for x in itertools.product((3, 5, 10, 15),repeat=3)]
len(grid_layers)

params = {"hidden_layer_sizes": grid_layers,
          "activation": ["identity", "logistic", "tanh", "relu"],
          "solver":["lbfgs", "sgd", "adam"],
          "alpha":[1e-3],
          "learning_rate":["constant", "invscaling", "adaptive"],
          "random_state":[101]}

grid = GridSearchCV(MLPRegressor(), params, verbose=2)
grid.fit(X_train, y_train)

print(grid.best_estimator_)
print(grid.best_params_)
res_grid = pd.DataFrame(grid.cv_results_)
res_grid.to_excel("data/res_grid_genesis.xlsx")

# Neural Net
# model, fit, predict
mlp_reg = MLPRegressor(**grid.best_params_)
mlp_reg.fit(X_train,y_train)
y_pred = mlp_reg.predict(X_test)

print("{} R^2, coef of determination of the prediction".format(mlp_reg.score(X_test, y_test)))

y_pred = pd.DataFrame(mlp_reg.predict(X))

y_reg = pd.concat([df_y, y_pred], axis=1).rename({0:"pred"}, axis=1)
print(y_reg)

%%time

df_ref = df_tract.copy(deep=True).loc[tracts.index, :]

# Drop Rows with missing values
df = df_ref.dropna(axis=0, how="any")

# Center and Scale
df_cs = pd.DataFrame(scale(df, axis=0),
                     index=df.index,
                     columns=df.columns)

# Train, X, on columns without missing values
df_x = df_cs[df_ref.dropna(axis=1, how="any").columns]

# Predict, y, columns with missing values
df_y = df_cs.drop(df_x.columns, axis=1)

print(df_ref.shape, df.shape, df_x.shape, df_y.shape)



X = df_x.values
y = df_y.values
# prep data: train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.1,
                                                    random_state=101)

grid_layers = [x for x in itertools.product((3, 5, 10, 15),repeat=3)]
len(grid_layers)

params = {"hidden_layer_sizes": grid_layers,
          "activation": ["identity", "logistic", "tanh", "relu"],
          "solver":["lbfgs", "sgd", "adam"],
          "alpha":[1e-3],
          "learning_rate":["constant", "invscaling", "adaptive"],
          "random_state":[101]}

grid = GridSearchCV(MLPRegressor(), params, verbose=2)
grid.fit(X_train, y_train)

print(grid.best_estimator_)
print(grid.best_params_)
res_grid = pd.DataFrame(grid.cv_results_)
res_grid.to_excel("data/res_grid_genesis.xlsx")

# Neural Net
# model, fit, predict
mlp_reg = MLPRegressor(**grid.best_params_)
mlp_reg.fit(X_train,y_train)
y_pred = mlp_reg.predict(X_test)

print("{} R^2, coef of determination of the prediction".format(mlp_reg.score(X_test, y_test)))

y_pred = pd.DataFrame(mlp_reg.predict(X))

y_reg = pd.concat([df_y, y_pred], axis=1).rename({0:"pred"}, axis=1)

svd = TruncatedSVD(n_components=3, n_iter=7, random_state=42)

X_svd = svd.fit_transform(df_x.values)

print(X_svd)

X = X_svd
y = df_y.values
# prep data: train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y,
                                                    test_size=0.3,
                                                    random_state=101)

# grid_layers = [x for x in itertools.product((10,20,30,40,50),repeat=4)]
# len(grid_layers)

grid_layers = [x for x in itertools.product((1, 2, 3, 4),repeat=3)]
len(grid_layers)

params = {"hidden_layer_sizes": grid_layers,
          "activation": ["tanh"],
          "solver":['sgd'],
          "alpha":[1e-4],
          "learning_rate":["invscaling"],
          "random_state":[101],
          "max_iter":[200, 500]}


grid = GridSearchCV(MLPRegressor(), params, verbose=2)
grid.fit(X_train, y_train)

print(grid.best_estimator_)
print(grid.best_params_)
res_grid = pd.DataFrame(grid.cv_results_)
res_grid.to_excel("data/res_grid_genesis.xlsx")


# Neural Net
# model, fit, predict
mlp_reg = MLPRegressor(**grid.best_params_)
mlp_reg.fit(X_train,y_train)
y_pred = mlp_reg.predict(X_test)

print("{} R^2, coef of determination of the prediction".format(mlp_reg.score(X_test, y_test)))
# print("Number of mislabeled points out of a total %d points : %d" % (X_test.shape[0], (y_test != y_pred).sum()))
# print("%d percent correct" % (100*float((y_test == y_pred).sum()/X_test.shape[0])))

# %% Final
# check = df_d.loc[df_d == 4].index
y_pred = pd.DataFrame(mlp_reg.predict(X))

y_reg = pd.concat([df_y, y_pred], axis=1).rename({0:"pred"}, axis=1)

y_reg.head()
y_reg.tail()